In [1]:
import geopandas as gpd
from folium import (
    LayerControl,
    Element,
    TileLayer,
)
import plotly.express as px
from os import path, makedirs



from core.downloads.geosampa import get_capabilities, get_features

# Promoção da Sustentabilidade Ambiental, Gestão de risco

## Formulário 7

O formulário associado a este notebook solicita os dados sobre 
`áreas de risco (hidrológico e deslizamentos)`. A justificativa para estes dados é a seguinte:

> As áreas de risco estão relacionadas à construção de moradias, em sua maioria em condições precárias, em locais com geológico-geotécnicas frágeis, não recomendadas para ocupação. O impacto de chuvas concentradas intensas, características de eventos climáticos extremos deve ser monitorado pelo Município, através de políticas públicas de gestão de risco e promoção da resiliência climática.

## Carregando as camadas de risco hidrológico e geológico

In [2]:
get_capabilities('hidrológico')

[{'name': 'geoportal:risco_hidrologico',
  'title': 'Risco Hidrológico',
  'abstract': 'Áreas de risco de enchentes e inundações em assentamentos precários.'}]

In [3]:
get_capabilities('deslizamento')

[{'name': 'geoportal:risco_ocorrencia_deslizamento',
  'title': 'Deslizamento',
  'abstract': 'Deslizamento.'},
 {'name': 'geoportal:area_risco_geologico',
  'title': 'Risco Geológico',
  'abstract': 'Polígonos que delimitam as áreas de risco geológico em assentamentos precários, sujeitas a deslizamentos e solapamento de margem de córrego no Município de São Paulo levantadas pelo IPT/SMSP entre 2009 e 2011.'},
 {'name': 'geoportal:risco_ocorrencia_risco_deslizamento',
  'title': 'Risco de Deslizamento',
  'abstract': 'Risco de deslizamento.'}]

Como o formulário cita apenas a camada `proteção e defesa civil/mapeamento de areas de risco`, assumirei que a referência a deslizamentos seja sobre a camada de risco geológico.

In [4]:
df_hid = get_features('geoportal:risco_hidrologico')
df_hid.head()

,id,cd_identificador_risco_hidrologico,nm_area_risco_hidrologico,tx_grau_risco_hidrologico,sg_area_risco_hidrologico,sg_setor_risco_hidrologico,tx_tipo_processo,dt_vistoria,qt_moradia,nm_subprefeitura,nm_bacia_hidrografica,geometry
0,risco_hidrologico.1,1,BARTOLOMEU FEIO,R2,HPI-01,HPI-01-01 (R2),ALAGAMENTO,2021-08-24,44,PINHEIROS,CORREGO AGUA ESPRAIADA,"POLYGON ((327424.845 7387319.281, 327417.205 7..."
1,risco_hidrologico.2,2,MAURO,R2,HVM-01,HVM-01-01 (R2),ENCHENTE/INUNDACAO,2022-02-01,15,VILA MARIANA,CORREGO UBERABA,"POLYGON ((332457.88 7386410.997, 332469.137 73..."
2,risco_hidrologico.3,3,MAURO,R3,HVM-01,HVM-01-02 (R3),ENCHENTE/INUNDACAO,2022-02-01,57,VILA MARIANA,CORREGO UBERABA,"POLYGON ((332476.102 7386409.838, 332469.228 7..."
3,risco_hidrologico.4,4,NEIDE APARECIDA SOLITO,R2,HVM-02,HVM-02-01 (R2),ALAGAMENTO,2022-02-01,12,VILA MARIANA,CORREGO DO SAPATEIRO,"POLYGON ((331840.472 7389826.482, 331845.268 7..."
4,risco_hidrologico.5,5,ESTEVAM HERNANDES,R2,HVM-03,HVM-03-01 (R2),ENCHENTE/INUNDACAO,2022-02-01,8,VILA MARIANA,CORREGO ACLIMACAO,"POLYGON ((333351.063 7391149.539, 333377.019 7..."


In [5]:
df_geo = get_features('geoportal:area_risco_geologico')
df_geo.head()

,id,cd_identificador,nm_area_risco,tx_grau_de_risco_geologico,sg_area_risco,sg_setor_risco,tx_tipo_processo_geologico,cd_tipo_processo_geologico,cd_grau_risco_geologico,dt_atualizacao,dt_vistoria,sg_fonte_original,qt_moradia,geometry
0,area_risco_geologico.1,1,AVENIDA SANTO AFONSO I,AREA ENCERRADA,AD-01,AREA ENCERRADA,ESCORREGAMENTO,2,99,2026-01-04 03:00:00+00:00,2022-02-02,18,0,"POLYGON ((332306.817 7380115.28, 332246.187 73..."
1,area_risco_geologico.2,2,RUA SAO BENTO XV,r3,AD-04,AD-04-03 (R3),SOLAPAMENTO,4,3,2026-01-04 03:00:00+00:00,2025-03-28,18,100,"POLYGON ((332324.725 7378882.633, 332309.159 7..."
2,area_risco_geologico.3,3,RUA SAO BENTO XV,r3,AD-04,AD-04-01 (R3),SOLAPAMENTO,4,3,2026-01-04 03:00:00+00:00,2025-03-28,18,30,"POLYGON ((332060.389 7379330.535, 332080.779 7..."
3,area_risco_geologico.4,4,RUA SAO BENTO XV,r1,AD-04,AD-04-02 (R1),SOLAPAMENTO,4,1,2026-01-04 03:00:00+00:00,2025-03-28,18,200,"POLYGON ((332309.159 7378872.908, 332286.448 7..."
4,area_risco_geologico.5,5,RUA SAO BENTO XV,r1,AD-04,AD-04-04 (R1),SOLAPAMENTO,4,1,2026-01-04 03:00:00+00:00,2025-03-28,18,250,"POLYGON ((332324.725 7378882.633, 332310.909 7..."


Além dos dados de riscos hidrogeológicos, também precisaremos dos dados do Censo de 2022 para a estimativa populacional. Os dados básicos, como número de domicílios e população, são disponibilizados diretamente no geopackage com as geometrias de setores censitários.

In [6]:
df_censo = gpd.read_file('https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/malha_com_atributos/setores/gpkg/UF/SP/SP_setores_CD2022.gpkg')
df_censo.head()

,CD_SETOR,SITUACAO,CD_SIT,CD_TIPO,AREA_KM2,CD_REGIAO,NM_REGIAO,CD_UF,NM_UF,CD_MUN,...,CD_CONCURB,NM_CONCURB,v0001,v0002,v0003,v0004,v0005,v0006,v0007,geometry
0,350010505000001,Urbana,1,0,0.123824,3,Sudeste,35,São Paulo,3500105,...,.,None,288,199,199,0,2.1,0.1667,138,"POLYGON ((-51.07255 -21.68865, -51.07237 -21.6..."
1,350010505000002,Urbana,1,0,0.202614,3,Sudeste,35,São Paulo,3500105,...,.,None,674,339,339,0,2.4,0.0143,280,"POLYGON ((-51.07378 -21.6862, -51.07388 -21.68..."
2,350010505000003,Urbana,1,0,0.235048,3,Sudeste,35,São Paulo,3500105,...,.,None,380,269,268,1,1.9,0.0404,198,"POLYGON ((-51.07138 -21.68313, -51.06965 -21.6..."
3,350010505000004,Urbana,1,0,0.288468,3,Sudeste,35,São Paulo,3500105,...,.,None,570,290,288,2,2.4,0.0127,236,"POLYGON ((-51.0688 -21.68808, -51.0673 -21.690..."
4,350010505000005,Urbana,1,0,0.213724,3,Sudeste,35,São Paulo,3500105,...,.,None,612,313,312,1,2.2,0.0404,272,"POLYGON ((-51.06969 -21.69442, -51.06928 -21.6..."


In [7]:
df_censo = df_censo.loc[df_censo['CD_MUN']=='3550308']
df_censo.head()

,CD_SETOR,SITUACAO,CD_SIT,CD_TIPO,AREA_KM2,CD_REGIAO,NM_REGIAO,CD_UF,NM_UF,CD_MUN,...,CD_CONCURB,NM_CONCURB,v0001,v0002,v0003,v0004,v0005,v0006,v0007,geometry
67060,355030801000001,Urbana,1,0,0.071797,3,Sudeste,35,São Paulo,3550308,...,3550308,São Paulo/SP,682,329,329,0,2.4,0.0381,289,"POLYGON ((-46.56954 -23.56918, -46.57016 -23.5..."
67061,355030801000002,Urbana,1,0,0.071902,3,Sudeste,35,São Paulo,3550308,...,3550308,São Paulo/SP,1374,1011,1011,0,2.3,0.0150,599,"POLYGON ((-46.56806 -23.56521, -46.56786 -23.5..."
67062,355030801000003,Urbana,1,0,0.055681,3,Sudeste,35,São Paulo,3550308,...,3550308,São Paulo/SP,557,239,238,1,2.7,0.0927,205,"POLYGON ((-46.56619 -23.56605, -46.56632 -23.5..."
67063,355030801000004,Urbana,1,0,0.064905,3,Sudeste,35,São Paulo,3550308,...,3550308,São Paulo/SP,526,245,245,0,2.4,0.0136,221,"POLYGON ((-46.56876 -23.56856, -46.56863 -23.5..."
67064,355030801000005,Urbana,1,0,0.086822,3,Sudeste,35,São Paulo,3550308,...,3550308,São Paulo/SP,579,258,258,0,2.6,0.0905,221,"POLYGON ((-46.57121 -23.57005, -46.57056 -23.5..."


## Carregando a camada de cobertura vegetal

In [8]:
get_capabilities('veg')

[{'name': 'geoportal:cobertura_vegetal',
  'title': 'Mapeamento da Cobertura Vegetal 2020',
  'abstract': 'Cobertura vegetal proveniente do Mapeamento da Vegetação.'},
 {'name': 'geoportal:GEOSAMPA_v_cgpabi_vegetacao_significativa',
  'title': 'Vegetação Significativa 2023',
  'abstract': 'Arborização Urbana'},
 {'name': 'geoportal:pde2014_v_parq_pde_map',
  'title': 'Áreas Verdes (5)',
  'abstract': 'Rede Hídrica Ambiental e Sistema de Áreas Protegidas, Áreas Verdes e Espaços Livres:\r\nO Sistema de Áreas Protegidas, Áreas Verdes e Espaços Livres é constituído pelo conjunto de áreas enquadradas nas diversas categorias protegidas pela legislação ambiental, de terras indígenas, de áreas prestadoras de serviços ambientais, das diversas tipologias de parques de logradouros públicos, de espaços vegetados e de espaços não ocupados por edificação coberta, de propriedade pública ou particular.'}]

In [9]:
df_veg = get_features('geoportal:cobertura_vegetal')
df_veg.head()


,id,cd_identificador_vegetacao,cd_identificador_original_vegetacao,cd_controle_vetor,tx_datum,cd_coordenada_x,cd_coordenada_y,cd_categoria_vegetacao,cd_subcategoria_vegetacao,cd_subcategoria_complementar_vegetacao,tx_descricao_categoria_subcategoria,qt_area_vegetacao,qt_perimetro_vegetacao,qt_altura_media_vegetacao,qt_desvio_altura_vegetacao,tx_data_voo,tx_entrega,tx_lote,tx_lote_versao,geometry
0,cobertura_vegetal.1,1,1,6007,"SIRGAS 2000,4 - 23S",332758.39,7397643.24,13,0.0,0,"Média a alta cobertura arbórea, arbóreo-arbust...",348.61758,120.94237,0.0,0.0,2017,04_2018,001,V2,"POLYGON ((332753.553 7397632.176, 332753.507 7..."
1,cobertura_vegetal.2,2,2,879,"SIRGAS 2000,4 - 23S",333723.72,7397905.09,13,0.0,0,"Média a alta cobertura arbórea, arbóreo-arbust...",5719.76786,1021.64520,0.0,0.0,2017,04_2018,001,V2,"POLYGON ((333701.991 7397891.395, 333704.261 7..."
2,cobertura_vegetal.3,3,3,92439,"SIRGAS 2000,4 - 23S",329698.03,7397184.05,14,0.0,0,Vegetação herbáceo-arbustiva,162.61484,76.32568,0.0,0.0,2017,04_2018,001,V2,"POLYGON ((329681.866 7397188.032, 329707.705 7..."
3,cobertura_vegetal.4,4,4,92440,"SIRGAS 2000,4 - 23S",331187.14,7398350.32,14,0.0,0,Vegetação herbáceo-arbustiva,132.22490,153.70879,0.0,0.0,2017,04_2018,001,V2,"POLYGON ((331152.165 7398355.317, 331226.746 7..."
4,cobertura_vegetal.5,5,5,92441,"SIRGAS 2000,4 - 23S",331186.11,7398338.59,14,0.0,0,Vegetação herbáceo-arbustiva,115.37687,164.42277,0.0,0.0,2017,04_2018,001,V2,"POLYGON ((331150.944 7398344.021, 331231.026 7..."


## Carregando a camada de quadras viárias

In [10]:
get_capabilities('viaria')

[{'name': 'geoportal:classificacao_viaria_cet',
  'title': 'Classificacao Viaria CET',
  'abstract': 'Classificação viária, de acordo com a portaria DSV.G 018/19.'},
 {'name': 'geoportal:zoneamento_classificacao_viaria',
  'title': 'Classificação Viária - Revogado - Lei 13.885/04',
  'abstract': 'Sistema viário classificado segundo as Leis nº 13.430/02 e 13.885/04, e as alterações decorrentes da aplicação da RESOLUÇÃO/SEMPLA/CTLU/023/2005. A relação das vias classificadas como estruturais ou coletoras foram associadas a base do Mapa Digital da Cidade (MDC).'},
 {'name': 'geoportal:classificacao_viaria_quadro_9',
  'title': 'Classificação viária – Quadro 9',
  'abstract': 'Classificação viária de acordo com o Quadro 09 da Lei 16.050/2014.'},
 {'name': 'geoportal:quadra_viaria_editada',
  'title': 'Quadra Viária',
  'abstract': 'As quadras viárias são polígonos fechados, normalmente gerados a partir da restituição dos meios-fios e das linhas de delimitação do leito carroçável. Nas escala

In [11]:
df_quadras = get_features('geoportal:quadra_viaria_editada')
df_quadras.head()


,id,cd_identificador_quadra_viaria_editada,cd_identificador_quadra_viaria_editada_original,tx_tipo_quadra_viaria,tx_escala,tx_ano_referencia,qt_area_metro,sg_fonte_original,geometry
0,quadra_viaria_editada.fid-4e630f15_19bd71081d1...,1,52943,Quadra,1:1.000,2004,7650,SMUL/GEOINFO,"POLYGON ((338344.482 7407584.661, 338344.543 7..."
1,quadra_viaria_editada.fid-4e630f15_19bd71081d1...,2,4,Praca_Canteiro,1:1.000,2004,11,SMUL/GEOINFO,"POLYGON ((327186.316 7377154.793, 327188.886 7..."
2,quadra_viaria_editada.fid-4e630f15_19bd71081d1...,3,5,Praca_Canteiro,1:1.000,2004,6,SMUL/GEOINFO,"POLYGON ((324545.386 7402131.104, 324544.17 74..."
3,quadra_viaria_editada.fid-4e630f15_19bd71081d1...,4,6,Praca_Canteiro,1:1.000,2004,109,SMUL/GEOINFO,"POLYGON ((326273.589 7399282.971, 326273.013 7..."
4,quadra_viaria_editada.fid-4e630f15_19bd71081d1...,5,7,Praca_Canteiro,1:1.000,2004,12,SMUL/GEOINFO,"POLYGON ((322149.447 7392535.031, 322154.771 7..."


## Carregando a camada de subprefeituras

In [12]:
get_capabilities('subprefeitura')

[{'name': 'geoportal:GEOSAMPA_v_praca_largo',
  'title': 'Cadastro de Praças e Largos',
  'abstract': 'Localização das praças e largos no território do município de São Paulo, com base nas informações repassadas por subprefeituras, SF e SEGES/CGPATRI.'},
 {'name': 'geoportal:perimetro_zoneamento_revogado_lei13885',
  'title': 'Perímetro Zona de Uso - Revogado - Lei 13.885/04',
  'abstract': 'A Lei de Zoneamento estabelece normas complementares ao Plano Diretor Estratégico, institui os Planos Regionais Estratégicos das Subprefeituras, dispõe sobre o parcelamento, disciplina e ordena o Uso e Ocupação do Solo do Município de São Paulo. Os perímetros de zona de uso foram vetorizados perante o estabelecido na Lei nº 13.885/2004, tendo como base cartográfica o Mapa Digital da Cidade (MDC).'},
 {'name': 'geoportal:sede_subprefeitura',
  'title': 'Subprefeituras',
  'abstract': 'Localização das sedes físicas das Subprefeituras da Cidade de São Paulo.'},
 {'name': 'geoportal:subprefeitura',
  '

In [13]:
df_subs = get_features('geoportal:subprefeitura')
df_subs.head()

,id,cd_identificador_subprefeitura,cd_subprefeitura,nm_subprefeitura,tx_escala,sg_fonte_original,dt_criacao,cd_tipo_discrepancia,dt_atualizacao,cd_usuario_atualizacao,sg_subprefeitura,qt_area_quilometro,qt_area_metro,geometry
0,subprefeitura.1,1,02,PIRITUBA-JARAGUA,1:5000,GEOGSG,2023-11-30,100200300,2025-12-23 20:33:41.278000+00:00,,PJ,55,55021021.42,"POLYGON ((318663.925 7404127.712, 318663.251 7..."
1,subprefeitura.2,2,03,FREGUESIA-BRASILANDIA,1:5000,GEOGSG,2023-11-30,100200300,2025-12-23 20:33:41.280000+00:00,,FO,32,31980202.74,"POLYGON ((327340.628 7399133.313, 327331.514 7..."
2,subprefeitura.3,3,04,CASA VERDE-LIMAO-CACHOEIRINHA,1:5000,GEOGSG,2023-11-30,100200300,2025-12-23 20:33:41.258000+00:00,,CV,27,27232342.53,"POLYGON ((329084.795 7402363.669, 329086.123 7..."
3,subprefeitura.4,4,05,SANTANA-TUCURUVI,1:5000,GEOGSG,2023-11-30,100200300,2025-12-23 20:33:41.284000+00:00,,ST,36,35782518.58,"POLYGON ((334076.366 7398045.594, 334074.986 7..."
4,subprefeitura.5,5,06,JACANA-TREMEMBE,1:5000,GEOGSG,2023-11-30,100200300,2025-12-23 20:33:41.257000+00:00,,JT,65,65115661.41,"POLYGON ((335167.648 7404409.048, 335167.247 7..."


## Ajustando as projeções

Vamos revisar os sistemas de coordenadas de todos os geodataframes para garantir que estão na mesma projeção.

In [14]:
for gdf in [df_geo, df_hid, df_censo, df_veg, df_quadras, df_subs]:
    print(gdf.columns[:5])
    print(gdf.crs)

Index(['id', 'cd_identificador', 'nm_area_risco', 'tx_grau_de_risco_geologico',
       'sg_area_risco'],
      dtype='object')
EPSG:31983
Index(['id', 'cd_identificador_risco_hidrologico', 'nm_area_risco_hidrologico',
       'tx_grau_risco_hidrologico', 'sg_area_risco_hidrologico'],
      dtype='object')
EPSG:31983
Index(['CD_SETOR', 'SITUACAO', 'CD_SIT', 'CD_TIPO', 'AREA_KM2'], dtype='object')
EPSG:4674
Index(['id', 'cd_identificador_vegetacao',
       'cd_identificador_original_vegetacao', 'cd_controle_vetor', 'tx_datum'],
      dtype='object')
EPSG:31983
Index(['id', 'cd_identificador_quadra_viaria_editada',
       'cd_identificador_quadra_viaria_editada_original',
       'tx_tipo_quadra_viaria', 'tx_escala'],
      dtype='object')
EPSG:31983
Index(['id', 'cd_identificador_subprefeitura', 'cd_subprefeitura',
       'nm_subprefeitura', 'tx_escala'],
      dtype='object')
EPSG:31983


Como o geodataframe do censo está em outro crs, precisamos convertê-lo para o `epsg:31983`.

In [15]:
df_censo = df_censo.to_crs('EPSG:31983')

In [16]:
df_geo.explore()

# Exportando os arquivos

Neste notebook, vamos apenas salvar os arquivos extraídos na pasta de entrada de dados.

In [17]:
output_dir = path.join('data', 'cache', 'urbanismo')

if not path.exists(output_dir):
    makedirs(output_dir)

for c, gdf in [('risco_hidrologico_original', df_hid),
               ('risco_geologico_original', df_geo),
               ('setores_censitarios_original', df_censo),
               ('cobertura_vegetal_original', df_veg),
               ('quadras_viarias_original', df_quadras),
               ('subprefeituras_original', df_subs)]:
    filename=path.join(output_dir, c)
    gdf.to_file(f'{filename}.gpkg', driver='GPKG', layer=c, overwrite=True)